# Module 14: Differential Geometry & Manifold Learning

Data in ML often lies on low-dimensional curved surfaces embedded in high-dimensional spaces (the **Manifold Hypothesis**). Furthermore, optimization over probability distributions can be viewed geometrically as moving along a curved statistical manifold. Differential geometry provides the mathematical tools to analyze these spaces.

## Contents
1. Manifolds, Atlases, and Coordinate Charts
2. Tangent Spaces and Vectors
3. Riemannian Metrics and Geodesics
4. Lie Groups and Lie Algebras (SO(3), SE(3))
5. Information Geometry & The Fisher Information Metric
6. Natural Gradient Descent

## 1. Manifolds, Atlases, and Coordinate Charts

A **differentiable manifold** $M$ of dimension $d$ is a topological space that locally resembles Euclidean space $\mathbb{R}^d$.

- **Coordinate Chart**: A pair $(U, \phi)$ where $U \subset M$ is open and $\phi: U \to \mathbb{R}^d$ is a homeomorphism.
- **Atlas**: A collection of charts $\{(U_i, \phi_i)\}$ that cover $M$.
- **Transition Map**: For overlapping charts, $\phi_j \circ \phi_i^{-1}: \mathbb{R}^d \to \mathbb{R}^d$ must be smooth ($C^\infty$). This allows us to do calculus on the manifold using local coordinates.

Common manifolds in ML include spheres ($S^n$), tori, the space of rotations ($SO(3)$), and statistical manifolds of probability distributions.

## 2. Tangent Spaces and Vectors

Since a manifold $M$ is curved, we cannot add two points $p, q \in M$ directly. Instead, at each point $p \in M$, we define a flat vector space called the **Tangent Space** $T_p M$, which contains all possible velocity vectors of curves passing through $p$.

The union of all tangent spaces is the **Tangent Bundle**:
$$TM = \bigcup_{p \in M} T_p M$$

## 3. Riemannian Metrics and Geodesics

To measure angles, lengths, and distances on a manifold, we need a **Riemannian Metric**. 

A Riemannian metric $g$ associates a positive-definite inner product $g_p: T_p M \times T_p M \to \mathbb{R}$ at each point $p$. In local coordinates $x^1, \dots, x^d$, the metric is represented as a symmetric positive-definite matrix $g_{ij}(x)$:
$$ds^2 = \sum_{i,j} g_{ij}(x) dx^i dx^j$$

### Geodesics
A **geodesic** is the generalization of a straight line to curved manifolds. It is a curve $\gamma(t)$ that minimizes path length between two points. Geodesics satisfy the geodesic differential equation:
$$\frac{d^2 \gamma^k}{dt^2} + \sum_{i,j} \Gamma_{ij}^k \frac{d\gamma^i}{dt} \frac{d\gamma^j}{dt} = 0$$
where $\Gamma_{ij}^k$ are the **Christoffel symbols** of the Levi-Civita connection, which describe how the coordinate basis changes across the manifold.

## 4. Lie Groups and Lie Algebras (SO(3), SE(3))

A **Lie Group** is a manifold that is also a group, where group multiplication and inversion are smooth operations.

- **$SO(3)$**: Special Orthogonal group, representing 3D rotations (matrices $R$ with $R^T R = I$ and $\det(R) = 1$).
- **$SE(3)$**: Special Euclidean group, representing 3D rigid body transformations (rotations + translations).

### Lie Algebra
The **Lie Algebra** $\mathfrak{g}$ is the tangent space of the Lie group at the identity element $e$. For $SO(3)$, the Lie algebra $\mathfrak{so}(3)$ is the space of $3 \times 3$ skew-symmetric matrices. The **exponential map** maps elements from the Lie algebra to the Lie group:
$$\exp: \mathfrak{g} \to G$$
For $SO(3)$, this is Rodrigues' formula for rotation matrices.

In [ ]:
from scipy.spatial.transform import Rotation as R
import numpy as np

# Exponentiate a skew-symmetric matrix (Lie algebra element) to get a rotation matrix (Lie group)
rot_vector = np.array([0.1, 0.2, 0.3])  # axis-angle representation
r = R.from_rotvec(rot_vector)
rotation_matrix = r.as_matrix()

print("Rotation Matrix SO(3):\n", rotation_matrix)
print("Determinant:", np.linalg.det(rotation_matrix))

## 5. Information Geometry & The Fisher Information Metric

Information geometry treats probability distributions as points on a statistical manifold. 

Let $p(x; \theta)$ be a family of distributions parameterized by $\theta$. The metric that measures the distance between distributions is the **Fisher Information Matrix (FIM)**:
$$F_{ij}(\theta) = \mathbb{E}_{x \sim p(x; \theta)} \left[ \frac{\partial \log p(x; \theta)}{\partial \theta_i} \frac{\partial \log p(x; \theta)}{\partial \theta_j} \right]$$

By Chentsov's Theorem, the FIM is the *unique* metric that is invariant under sufficient statistics. The distance induced by this metric is locally equivalent to the KL divergence:
$$D_{KL}(p(x; \theta) \parallel p(x; \theta + d\theta)) \approx \frac{1}{2} d\theta^T F(\theta) d\theta$$

## 6. Natural Gradient Descent

Standard gradient descent updates parameters using Euclidean distance in parameter space:
$$\theta_{k+1} = \theta_k - \alpha \nabla J(\theta_k)$$

However, a small change in $\theta$ can lead to a massive change in the output distribution $p(x; \theta)$ (e.g. changing the variance of a Gaussian near zero). 

**Natural Gradient Descent** updates parameters by moving along the steepest descent direction on the statistical manifold, constraining the step size using the KL divergence instead of Euclidean distance. The update is:
$$\theta_{k+1} = \theta_k - \alpha F(\theta_k)^{-1} \nabla J(\theta_k)$$

This is heavily used in reinforcement learning (TRPO, PPO) and optimization of deep networks (K-FAC).